# ClauseDraft — hosted QLoRA run

The full epoch runs here, on a free Colab GPU, instead of an 8 GB laptop. In the menu, choose **Runtime → Change runtime type → T4 GPU** before running the cells.

The runtime disk is wiped when the session ends. Checkpoints and the finished adapter are written to Google Drive, so a dropped session can be continued.

This notebook does not install `requirements.txt`. That file pins a CUDA 13.0 torch wheel for an RTX 50-series laptop. Colab already has a CUDA torch, and replacing it breaks the GPU.

In [1]:
import shutil
import torch

if shutil.which("nvidia-smi") is None or not torch.cuda.is_available():
    raise SystemExit("No GPU in this runtime. Change the runtime type to a T4 and run this cell again.")
print(torch.cuda.get_device_name(0))

Tesla T4


In [2]:
import os
import subprocess
from pathlib import Path

root = Path("/content/domain-specific-llm-fine-tuning")
if not (root / "scripts" / "train.py").is_file():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/Armiin-h/domain-specific-llm-fine-tuning.git",
        str(root),
    ])
os.chdir(root)
print(root)

/content/domain-specific-llm-fine-tuning


In [3]:
import subprocess
import sys
import torch

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
import torch as torch_after
if not torch_after.cuda.is_available():
    raise SystemExit("The install replaced torch with a CPU build. Restart the runtime and run the cells again.")
print("torch", torch.__version__, "->", torch_after.__version__)

torch 2.11.0+cu130 -> 2.11.0+cu130


In [4]:
from google.colab import drive

drive.mount("/content/drive")
adapter_dir = "/content/drive/MyDrive/clausedraft-adapter"
print(adapter_dir)

Mounted at /content/drive
/content/drive/MyDrive/clausedraft-adapter


In [5]:
import subprocess
import sys

subprocess.check_call([sys.executable, "scripts/prepare_corpus.py"])
subprocess.check_call([sys.executable, "scripts/build_jsonl.py"])

0

In [6]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "scripts/train.py",
    "--output-dir",
    "/content/drive/MyDrive/clausedraft-adapter",
])

CalledProcessError: Command '['/usr/bin/python3', 'scripts/train.py', '--output-dir', '/content/drive/MyDrive/clausedraft-adapter']' returned non-zero exit status 1.

If the session stops after a checkpoint, mount Drive again and run the next cell. It continues from the last checkpoint in the Drive folder. A finished run leaves `adapter_model.safetensors`, `loss.svg`, and `model_card.md` in that folder. Download those for the eval and the API. The adapter is not committed to git.

In [7]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "scripts/train.py",
    "--resume",
    "--output-dir",
    "/content/drive/MyDrive/clausedraft-adapter",
])

CalledProcessError: Command '['/usr/bin/python3', 'scripts/train.py', '--resume', '--output-dir', '/content/drive/MyDrive/clausedraft-adapter']' returned non-zero exit status 1.